# Phase 2: Walker-Walk DMControl Results Analysis

## Overview

This notebook presents the results from **Phase 2** of the quantum-enhanced world model experiments on the **Walker-Walk** environment from the DeepMind Control Suite.

### Environment Details
- **Domain:** Walker
- **Task:** Walk
- **Observation Dimension:** 24
- **Action Dimension:** 6

### Approaches Compared
1. **Baseline** - Classical DreamerV3-style world model
2. **Quantum Tunneling** - Enhanced optimization to escape local minima
3. **Superposition** - Parallel exploration of multiple training paths
4. **Entanglement** - Correlated feature learning
5. **Interference Ensemble** - Quantum-inspired ensemble with interference patterns

### Configuration
- **Seeds:** [42, 123, 456, 789, 1024]
- **Training Steps:** 10,000
- **Architecture:** stoch_dim=64, deter_dim=512, hidden_dim=512

---

## 1. Imports

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt

# Set project root
project_root = Path.cwd().parent

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

---

## 2. Load Results

In [2]:
# Load the complete metrics JSON
results_path = project_root / 'experiments' / 'results' / 'phase2' / 'walker' / 'complete_metrics.json'

with open(results_path, 'r') as f:
    data = json.load(f)

# Extract components
experiment_name = data['experiment']
env_info = data['environment']
config = data['config']
summary = data['summary']
raw_results = data['raw_results']

# Print overview
print(f"Experiment: {experiment_name}")
print(f"Environment: {env_info['domain']} / {env_info['task']}")
print(f"Observation Dim: {env_info['obs_dim']}, Action Dim: {env_info['action_dim']}")
print(f"\nConfiguration:")
print(f"  - Stochastic Dim: {config['stoch_dim']}")
print(f"  - Deterministic Dim: {config['deter_dim']}")
print(f"  - Hidden Dim: {config['hidden_dim']}")
print(f"  - Batch Size: {config['batch_size']}")
print(f"  - Sequence Length: {config['seq_len']}")
print(f"  - Training Steps: {config['num_steps']}")
print(f"  - Learning Rate: {config['learning_rate']}")
print(f"  - Seeds: {config['seeds']}")
print(f"\nApproaches evaluated: {len(summary)}")
print(f"Total runs: {len(raw_results)}")

Experiment: phase2_walker_walk
Environment: walker / walk
Observation Dim: 24, Action Dim: 6

Configuration:
  - Stochastic Dim: 64
  - Deterministic Dim: 512
  - Hidden Dim: 512
  - Batch Size: 32
  - Sequence Length: 20
  - Training Steps: 10000
  - Learning Rate: 0.0003
  - Seeds: [42, 123, 456, 789, 1024]

Approaches evaluated: 5
Total runs: 25


---

## 3. Results Summary Table

In [3]:
# Create formatted results table
approaches = ['baseline', 'quantum_tunneling', 'superposition', 'entanglement', 'interference_ensemble']
approach_display = {
    'baseline': 'baseline',
    'quantum_tunneling': 'quantum_tunneling',
    'superposition': 'superposition',
    'entanglement': 'entanglement',
    'interference_ensemble': 'interference_ensemble'
}

# Sort by test MSE (ascending)
sorted_approaches = sorted(approaches, key=lambda x: summary[x]['test_obs_mse_mean'])

print("\n" + "="*80)
print("                    WALKER-WALK WORLD MODEL RESULTS SUMMARY")
print("="*80)
print(f"\n{'Approach':>30}  {'Test MSE (Mean +/- Std)':>23}  {'Train MSE':>9}  {'Time (s)':>8}  {'Parameters':>12}")
print("-"*80)

best_approach = sorted_approaches[0]
baseline_mse = summary['baseline']['test_obs_mse_mean']

for approach in sorted_approaches:
    s = summary[approach]
    is_best = approach == best_approach
    label = f"{approach_display[approach]} (BEST)" if is_best else approach_display[approach]
    mse_str = f"{s['test_obs_mse_mean']:.3f} +/- {s['test_obs_mse_std']:.3f}"
    print(f"{label:>30}  {mse_str:>23}  {s['train_obs_mse_mean']:>9.3f}  {s['time_mean']:>8.1f}  {s['num_params']:>12,}")

print("-"*80)

# Best approach summary
best_mse = summary[best_approach]['test_obs_mse_mean']
improvement = (baseline_mse - best_mse) / baseline_mse * 100
print(f"\nBest Performing: {best_approach}")
print(f"  - Test MSE: {best_mse:.3f} +/- {summary[best_approach]['test_obs_mse_std']:.3f}")
print(f"  - Improvement over baseline: {improvement:.1f}%")


                    WALKER-WALK WORLD MODEL RESULTS SUMMARY

                       Approach  Test MSE (Mean +/- Std)  Train MSE  Time (s)  Parameters
--------------------------------------------------------------------------------
  interference_ensemble (BEST)        1.022 +/- 0.013      0.820     3102.7    23,783,816
            quantum_tunneling        1.797 +/- 0.030      1.572     1533.7     4,756,762
                  entanglement        1.798 +/- 0.032      1.543     1349.5     5,282,714
                      baseline        1.799 +/- 0.060      1.543     1585.7     4,756,762
                  superposition        4.645 +/- 0.198      4.632     1327.6     4,756,762
--------------------------------------------------------------------------------

Best Performing: interference_ensemble
  - Test MSE: 1.022 +/- 0.013
  - Improvement over baseline: 43.2%


---

## 4. Statistical Analysis

In [4]:
# Extract per-seed results for statistical tests
def get_approach_values(approach_name):
    """Get test MSE values for all seeds of an approach."""
    return [r['test_obs_mse'] for r in raw_results if r['approach'] == approach_name]

baseline_values = get_approach_values('baseline')

print("\n" + "="*80)
print("                         STATISTICAL ANALYSIS VS BASELINE")
print("="*80)
print("\nComparison Method: Mann-Whitney U Test (non-parametric)")
print("Effect Size: Cohen's d")
print("Significance Level: alpha = 0.05")
print(f"\n{'Approach':>25}  {'Mean MSE':>10}  {'Diff vs BL':>10}  {'Mann-Whitney U':>14}  {'p-value':>9}  {'Cohen\'s d':>9}  {'Significant':>11}")
print("-"*96)

def cohens_d(group1, group2):
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0

significant_improvements = []
significant_degradations = []

for approach in ['quantum_tunneling', 'superposition', 'entanglement', 'interference_ensemble']:
    values = get_approach_values(approach)
    
    # Mann-Whitney U test
    u_stat, p_value = stats.mannwhitneyu(values, baseline_values, alternative='two-sided')
    
    # Cohen's d
    d = cohens_d(values, baseline_values)
    
    # Significance
    is_significant = p_value < 0.05
    sig_str = "YES" if is_significant else "No"
    
    # Difference from baseline
    mean_diff = np.mean(values) - np.mean(baseline_values)
    diff_str = f"+{mean_diff:.3f}" if mean_diff > 0 else f"{mean_diff:.3f}"
    
    print(f"{approach:>25}  {np.mean(values):>10.3f}  {diff_str:>10}  {u_stat:>14.1f}  {p_value:>9.4f}  {d:>9.2f}  {sig_str:>11}")
    
    if is_significant and mean_diff < 0:
        significant_improvements.append((approach, p_value, d))
    elif is_significant and mean_diff > 0:
        significant_degradations.append((approach, p_value, d))

print("-"*96)

# Print key findings
print("\nKEY FINDINGS:")
for approach, p, d in significant_improvements:
    print(f"  [*] {approach} shows SIGNIFICANT improvement (p={p:.3f}, d={d:.2f})")
for approach, p, d in significant_degradations:
    print(f"  [!] {approach} shows SIGNIFICANT degradation (p={p:.3f}, d={d:.2f})")


                         STATISTICAL ANALYSIS VS BASELINE

Comparison Method: Mann-Whitney U Test (non-parametric)
Effect Size: Cohen's d
Significance Level: alpha = 0.05

                  Approach   Mean MSE  Diff vs BL  Mann-Whitney U   p-value  Cohen's d  Significant
------------------------------------------------------------------------------------------------
         quantum_tunneling      1.797      -0.002           12.0    0.9168     -0.034           No
             superposition      4.645      +2.846            0.0    0.0079      19.44          YES
               entanglement      1.798      -0.001           12.0    0.9168     -0.032           No
    interference_ensemble      1.022      -0.777            0.0    0.0079     -17.98          YES
------------------------------------------------------------------------------------------------

KEY FINDINGS:
  [*] interference_ensemble shows SIGNIFICANT improvement (p=0.008, d=-17.98)
  [!] superposition shows SIGNIFICANT degrad

---

## 5. Visualization: Performance Comparison

In [5]:
# Bar chart comparing test MSE across approaches
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data
display_names = ['Baseline', 'Quantum\nTunneling', 'Superposition', 'Entanglement', 'Interference\nEnsemble']
means = [summary[a]['test_obs_mse_mean'] for a in approaches]
stds = [summary[a]['test_obs_mse_std'] for a in approaches]

# Colors: green for best, red for worst, blue for baseline, gray for others
colors = ['#3498db', '#95a5a6', '#e74c3c', '#95a5a6', '#27ae60']

# Create bars
x = np.arange(len(approaches))
bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for i, (bar, mean, std) in enumerate(zip(bars, means, stds)):
    height = bar.get_height()
    ax.annotate(f'{mean:.3f}\n+/-{std:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height + std + 0.1),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add baseline reference line
ax.axhline(y=summary['baseline']['test_obs_mse_mean'], color='#3498db', 
           linestyle='--', linewidth=2, alpha=0.7, label='Baseline Reference')

# Formatting
ax.set_xlabel('Approach', fontsize=12)
ax.set_ylabel('Test Observation MSE (lower is better)', fontsize=12)
ax.set_title('Walker-Walk: World Model Prediction Error by Approach', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(display_names, fontsize=11)
ax.legend(loc='upper right')
ax.set_ylim(0, max(means) + max(stds) + 0.8)

# Add significance markers
ax.annotate('** p<0.01', xy=(4, means[4] - 0.15), ha='center', fontsize=9, color='#27ae60', fontweight='bold')
ax.annotate('** p<0.01', xy=(2, means[2] + stds[2] + 0.4), ha='center', fontsize=9, color='#e74c3c', fontweight='bold')

plt.tight_layout()
plt.savefig(project_root / 'phase2_dmcontrol_notebooks' / 'walker_walk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. Per-Seed Results Boxplot

In [6]:
# Boxplot showing distribution across seeds
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data for boxplot
boxplot_data = [get_approach_values(a) for a in approaches]

# Create boxplot
bp = ax.boxplot(boxplot_data, labels=display_names, patch_artist=True)

# Color the boxes
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add individual points
for i, (approach, data) in enumerate(zip(approaches, boxplot_data)):
    x_jitter = np.random.normal(i+1, 0.04, size=len(data))
    ax.scatter(x_jitter, data, alpha=0.6, color='black', s=50, zorder=5)

# Add baseline reference line
ax.axhline(y=summary['baseline']['test_obs_mse_mean'], color='#3498db', 
           linestyle='--', linewidth=2, alpha=0.7, label='Baseline Mean')

# Formatting
ax.set_xlabel('Approach', fontsize=12)
ax.set_ylabel('Test Observation MSE', fontsize=12)
ax.set_title('Walker-Walk: Per-Seed Results Distribution', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')

# Add seed count annotation
ax.annotate(f'n=5 seeds per approach', xy=(0.02, 0.98), xycoords='axes fraction',
            ha='left', va='top', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig(project_root / 'phase2_dmcontrol_notebooks' / 'walker_walk_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 7. Key Findings

### Performance Ranking (Test Observation MSE)

| Rank | Approach | Test MSE | vs Baseline |
|------|----------|----------|-------------|
| 1 | **Interference Ensemble** | 1.022 +/- 0.013 | **-43.2%** (p=0.008) |
| 2 | Quantum Tunneling | 1.797 +/- 0.030 | -0.1% (n.s.) |
| 3 | Entanglement | 1.798 +/- 0.032 | -0.1% (n.s.) |
| 4 | Baseline | 1.799 +/- 0.060 | - |
| 5 | Superposition | 4.645 +/- 0.198 | **+158.2%** (p=0.008) |

### Key Observations

1. **Interference Ensemble is the clear winner:**
   - Achieves 43.2% lower prediction error than baseline
   - Statistically significant (p=0.008)
   - Most consistent performance (lowest std deviation)
   - Trade-off: ~5x more parameters and ~2x training time

2. **Quantum Tunneling and Entanglement match baseline:**
   - No significant difference from classical approach
   - Slightly more consistent (lower variance)
   - Minimal computational overhead

3. **Superposition performs poorly:**
   - Significantly worse than baseline (p=0.008)
   - 2.5x higher prediction error
   - The parallel path exploration may interfere with convergence

### Computational Cost Analysis

| Approach | Parameters | Training Time | MSE/Time Efficiency |
|----------|------------|---------------|---------------------|
| Baseline | 4.76M | 1585.7s | 1.134e-3 |
| Quantum Tunneling | 4.76M | 1533.7s | 1.172e-3 |
| Superposition | 4.76M | 1327.6s | 3.499e-3 |
| Entanglement | 5.28M | 1349.5s | 1.332e-3 |
| Interference Ensemble | 23.78M | 3102.7s | 3.294e-4 |

---

## 8. Conclusion

In [7]:
# Print final summary
print("\n" + "="*80)
print("                    WALKER-WALK EXPERIMENT CONCLUSIONS")
print("="*80)

print(f"\nBEST APPROACH: Interference Ensemble")
print(f"  - Test MSE: {summary['interference_ensemble']['test_obs_mse_mean']:.3f} +/- {summary['interference_ensemble']['test_obs_mse_std']:.3f}")
print(f"  - Improvement: {improvement:.1f}% lower error than baseline")
print(f"  - Statistical Significance: p = 0.008 (Mann-Whitney U)")
print(f"  - Effect Size: Cohen's d = -17.98 (very large)")

print("\nRECOMMENDATIONS:")
print("  [1] For best prediction accuracy: Use Interference Ensemble")
print("      (accepts ~5x parameter increase and ~2x training time)")
print("  ")
print("  [2] For balanced performance: Use Quantum Tunneling or Entanglement")
print("      (matches baseline with slightly better consistency)")
print("  ")
print("  [3] Avoid Superposition for this environment")
print("      (significantly degrades performance)")

print("\nNEXT STEPS:")
print("  - Validate findings on Cheetah-Run environment")
print("  - Investigate why Superposition underperforms")
print("  - Analyze Interference Ensemble's parameter efficiency")
print("\n" + "="*80)


                    WALKER-WALK EXPERIMENT CONCLUSIONS

BEST APPROACH: Interference Ensemble
  - Test MSE: 1.022 +/- 0.013
  - Improvement: 43.2% lower error than baseline
  - Statistical Significance: p = 0.008 (Mann-Whitney U)
  - Effect Size: Cohen's d = -17.98 (very large)

RECOMMENDATIONS:
  [1] For best prediction accuracy: Use Interference Ensemble
      (accepts ~5x parameter increase and ~2x training time)
  
  [2] For balanced performance: Use Quantum Tunneling or Entanglement
      (matches baseline with slightly better consistency)
  
  [3] Avoid Superposition for this environment
      (significantly degrades performance)

NEXT STEPS:
  - Validate findings on Cheetah-Run environment
  - Investigate why Superposition underperforms
  - Analyze Interference Ensemble's parameter efficiency

